# 🧪 Laboratorio Clase 5 · Tu primer asistente con IA

**Módulo II – Navegando por los componentes de la IA · Sesión 5**

En 20 minutos vas a hablarle a un LLM (Claude, de Anthropic) **desde código**. Cuatro misiones:

| Misión | Qué vas a ver | Concepto de la clase |
|---|---|---|
| 1 · Cuenta tokens | Cómo "lee" un LLM y cuánto cuesta | Tokens |
| 2 · Juega con la temperatura | Respuestas predecibles vs. creativas | Temperatura |
| 3 · Clasificador CRF + few-shot | De texto libre a datos ordenados (JSON) | CRF, Few-Shot, Instrucciones negativas |
| 4 · Bonus: tu mini app | Un formulario que cualquiera puede usar | De observador a constructor |

### Cómo usar este notebook
- Ejecuta cada celda con **Shift + Enter** (o con ▶). Lee el resultado antes de pasar a la siguiente.
- El kernel debe ser el entorno del proyecto (`.venv`). Ahí ya están `anthropic`, `python-dotenv` y `tiktoken`.
- Si algo falla: **Kernel → Restart y Run All**.
- No necesitas entender cada línea de código: lee los comentarios que empiezan con `#`.


## Paso 0 · Conectar con Claude

La clave ya está en el archivo `.env` del proyecto, en la variable **`ANTHROPIC_API_KEY`**. Este notebook la carga solo. No la imprimas ni la pegues en el chat.

Usamos **Claude Haiku 4.5** (`claude-haiku-4-5`). Es un modelo rápido y todavía acepta el parámetro de temperatura. En la API actual (versión `2026-10-21`, la que trae el SDK instalado) Anthropic retiró ese parámetro para todos los modelos. Cuando jugamos con la temperatura, el notebook pide la versión anterior (`2023-06-01`), que Haiku 4.5 aún respeta. Si ese modelo no está disponible, el notebook pasa a **Haiku 5.5**, más barato y sin perilla de temperatura.

> **Gobernanza:** no pegues datos reales de clientes ni información confidencial. En este laboratorio todos los reclamos son inventados.


In [14]:
# Paso 0: conectar con Claude (ejecuta y espera el visto bueno)
import os, json, re, time
from pathlib import Path

import anthropic
from dotenv import load_dotenv

def cargar_env():
    aqui = Path.cwd().resolve()
    for carpeta in [aqui, *list(aqui.parents)[:5]]:
        env = carpeta / ".env"
        if env.is_file():
            load_dotenv(env)
            return True
    return False

def recortar(error, secreto):
    texto = str(error)
    if secreto:
        texto = texto.replace(secreto, "[clave]")
    return texto[:140]

cargo_env = cargar_env()
clave = os.environ.get("ANTHROPIC_API_KEY")
cliente, MODELO, ACEPTA_TEMPERATURA = None, None, False

# Haiku 4.5 acepta temperatura en la versión de API 2023-06-01.
# Haiku 5.5 es el respaldo: más barato, sin ese parámetro.
MODELOS = [
    ("claude-haiku-4-5", True),
    ("claude-haiku-4-5-20251001", True),
    ("claude-haiku-5-5", False),
]
VERSION_CON_TEMPERATURA = "2023-06-01"

if not cargo_env:
    print("No encontré un archivo .env desde esta carpeta hacia arriba.")
elif not clave:
    print("No encontré ANTHROPIC_API_KEY en el .env. Revísala y vuelve a ejecutar esta celda.")
else:
    cliente = anthropic.Anthropic()
    for candidato, acepta_temperatura in MODELOS:
        try:
            cliente.messages.create(
                model=candidato,
                max_tokens=16,
                messages=[{"role": "user", "content": "Responde solo: OK"}],
            )
            MODELO, ACEPTA_TEMPERATURA = candidato, acepta_temperatura
            break
        except Exception as error:
            print(f"   (no se pudo usar {candidato}: {recortar(error, clave)})")

if MODELO:
    perilla = "con control de temperatura" if ACEPTA_TEMPERATURA else "sin control de temperatura"
    print(f"Conectado a Claude. Modelo: {MODELO} ({perilla}).")
else:
    print("No hay conexión. Revisa ANTHROPIC_API_KEY en el archivo .env.")

def preguntar(prompt, temperatura=None, instrucciones=None, json_salida=False, reintentos=3, max_tokens=1024):
    # Envía un prompt a Claude y devuelve el texto de la respuesta.
    if cliente is None or MODELO is None:
        raise RuntimeError("Primero ejecuta el Paso 0 y consigue la conexión.")
    if json_salida:
        prompt = prompt + "\n\nResponde únicamente con JSON válido, sin texto alrededor."
    kwargs = {
        "model": MODELO,
        "max_tokens": max_tokens,
        "messages": [{"role": "user", "content": prompt}],
    }
    if instrucciones:
        kwargs["system"] = instrucciones
    if temperatura is not None and ACEPTA_TEMPERATURA:
        kwargs["extra_body"] = {"temperature": temperatura}
        kwargs["extra_headers"] = {"anthropic-version": VERSION_CON_TEMPERATURA}
    for _ in range(reintentos):
        try:
            respuesta = cliente.messages.create(**kwargs)
            return "".join(bloque.text for bloque in respuesta.content if bloque.type == "text").strip()
        except anthropic.APIStatusError as error:
            mensaje = str(error).lower()
            if error.status_code == 400 and "temperature" in mensaje and "extra_body" in kwargs:
                print("   La API rechazó la temperatura. Sigo sin ese parámetro.")
                kwargs.pop("extra_body", None)
                kwargs.pop("extra_headers", None)
                continue
            if error.status_code in (429, 529) or "overloaded" in mensaje:
                print("   Límite o saturación. Espero 20 segundos y reintento...")
                time.sleep(20)
                continue
            raise
    raise RuntimeError("Claude no respondió después de varios intentos.")


Conectado a Claude. Modelo: claude-haiku-4-5 (con control de temperatura).


## Misión 1 · Cuenta tokens 🔢
Un LLM no lee palabras: lee **tokens** (pedazos de palabras). Así se mide el costo: **tokens de entrada + tokens de salida = factura**.

Para *ver* los cortes usamos el tokenizador de OpenAI (`o200k_base`). Cada proveedor corta distinto. Al final le preguntamos a Claude cuántos tokens cuenta él para el mismo texto.


In [15]:
# Misión 1a: ver cómo se corta un texto en tokens
try:
    import tiktoken
    tokenizador = tiktoken.get_encoding("o200k_base")
except Exception as error:
    tokenizador = None
    print("No se pudo cargar el tokenizador de OpenAI:", str(error)[:80])

def ver_tokens(texto):
    if tokenizador is None:
        return None
    ids = tokenizador.encode(texto)
    print(f"{len(ids):>3} tokens →  " + " | ".join(tokenizador.decode([i]) for i in ids))
    return len(ids)

ver_tokens("ChatGPT")
ver_tokens("Inteligencia artificial")
ver_tokens("Artificial intelligence")

reclamo = "Hola, me cobraron dos veces la misma compra con mi tarjeta y nadie me responde en el call center."
ver_tokens(reclamo)

if cliente is not None and MODELO is not None:
    conteo = cliente.messages.count_tokens(
        model=MODELO,
        messages=[{"role": "user", "content": reclamo}],
    )
    print(f"\nClaude cuenta {conteo.input_tokens} tokens para el mismo reclamo (cada modelo corta distinto).")


  2 tokens →  Chat | GPT
  4 tokens →  Int | elig | encia |  artificial
  2 tokens →  Artificial |  intelligence
 22 tokens →  Hola | , |  me |  cobr | aron |  dos |  veces |  la |  misma |  compra |  con |  mi |  tarjeta |  y |  nadie |  me |  responde |  en |  el |  call |  center | .

Claude cuenta 39 tokens para el mismo reclamo (cada modelo corta distinto).


In [16]:
# Misión 1b: ¿cuánto costaría procesar UN MILLÓN de reclamos?
# Precios publicados de Claude Haiku, en USD por millón de tokens.
# Fuente: https://platform.claude.com/docs/en/about-claude/pricing (consulta de noviembre de 2026).
# Revisa siempre la página: los precios cambian.
PRECIOS = {
    "claude-haiku-4-5": (1.00, 5.00),
    "claude-haiku-4-5-20251001": (1.00, 5.00),
    "claude-haiku-5-5": (0.10, 0.50),  # prompts de hasta 100.000 tokens
}
PRECIO_ENTRADA, PRECIO_SALIDA = PRECIOS.get(MODELO, (1.00, 5.00))
TOKENS_INSTRUCCIONES = 400  # el prompt con rol, contexto, formato y ejemplos
TOKENS_RESPUESTA = 60       # un JSON corto

tokens_reclamo = (ver_tokens(reclamo) if tokenizador else 25)
n_reclamos = 1_000_000
entrada = n_reclamos * (tokens_reclamo + TOKENS_INSTRUCCIONES)
salida = n_reclamos * TOKENS_RESPUESTA
costo = entrada / 1e6 * PRECIO_ENTRADA + salida / 1e6 * PRECIO_SALIDA
print(f"\nModelo de la estimación: {MODELO}")
print(f"Precio: USD {PRECIO_ENTRADA:.2f} por millón de tokens de entrada, USD {PRECIO_SALIDA:.2f} de salida.")
print(f"Tokens de entrada: {entrada:,}".replace(",", "."))
print(f"Tokens de salida:  {salida:,}".replace(",", "."))
print(f"Costo estimado: USD {costo:,.0f}".replace(",", "."))
print("\n👉 Prueba: sube TOKENS_INSTRUCCIONES a 3000 (un prompt larguísimo). ¿Qué pasa con el costo?")


 22 tokens →  Hola | , |  me |  cobr | aron |  dos |  veces |  la |  misma |  compra |  con |  mi |  tarjeta |  y |  nadie |  me |  responde |  en |  el |  call |  center | .

Modelo de la estimación: claude-haiku-4-5
Precio: USD 1.00 por millón de tokens de entrada, USD 5.00 de salida.
Tokens de entrada: 422.000.000
Tokens de salida:  60.000.000
Costo estimado: USD 722

👉 Prueba: sube TOKENS_INSTRUCCIONES a 3000 (un prompt larguísimo). ¿Qué pasa con el costo?


## Misión 2 · Juega con la temperatura 🌡️
Mismo prompt, tres intentos con temperatura **baja (0.1)** y tres con temperatura **alta (1.0)**. ¿Cuál varía más?

En Claude Haiku 4.5 la temperatura sigue funcionando, pero solo si la petición usa la versión de API anterior a octubre de 2026. El Paso 0 ya deja eso resuelto. En modelos más nuevos (Haiku 5.5, Sonnet 5 y posteriores) Anthropic retiró la perilla: la variabilidad se controla con el prompt, no con un número.


In [17]:
# Misión 2: mismo prompt, distinta temperatura
prompt_slogan = ("Escribe UN slogan (máximo 8 palabras) para una nueva bebida energética chilena. "
                 "Responde solo con el slogan, sin comillas.")

if not ACEPTA_TEMPERATURA:
    print("ℹ️ Este modelo no acepta temperatura: verás la respuesta por defecto.\n")

for temperatura in [0.1, 1.0]:
    print(f"🌡️ Temperatura {temperatura}")
    for intento in range(1, 4):
        print(f"   {intento}. {preguntar(prompt_slogan, temperatura=temperatura, max_tokens=64)}")
    print()


🌡️ Temperatura 0.1
   1. Energía chilena que te lleva al límite
   2. Energía chilena que te lleva al límite
   3. Energía chilena que te lleva al límite

🌡️ Temperatura 1.0
   1. Energía chilena que te lleva al límite
   2. Energía de Chile, poder sin límites
   3. Energía chilena, potencia para conquista



## Misión 3 · Clasificador de reclamos: de *zero-shot* a CRF + *few-shot* 🏦
Un banco recibe miles de reclamos al día y necesita derivarlos al área correcta. Primero lo pedimos "a la rápida" (zero-shot); después, con el framework **CRF** (Contexto, Rol, Formato), **ejemplos** (few-shot) e **instrucciones negativas**.

In [18]:
# Misión 3a: los reclamos (todos inventados)
reclamos = [
    "Me cobraron dos veces la misma compra en el supermercado con mi tarjeta de débito. Necesito que me devuelvan la plata.",
    "Hace una semana pedí cerrar mi cuenta corriente y me siguen cobrando la mantención.",
    "Recibí un SMS del banco pidiendo mi clave dinámica y ahora veo transferencias que no hice. ¡Urgente!",
    "La app no me deja entrar desde la última actualización, me sale error 503.",
    "Quiero saber por qué me rechazaron el crédito de consumo si tengo buen historial.",
]

# ZERO-SHOT: pedimos sin contexto, sin rol, sin formato
lista = "\n".join(f"{i}. {r}" for i, r in enumerate(reclamos, 1))
print(preguntar("Clasifica estos reclamos:\n" + lista))

# Clasificación de Reclamos Bancarios

| # | Reclamo | Clasificación | Prioridad | Tipo |
|---|---------|----------------|-----------|------|
| **1** | Doble cobro en supermercado | **Fraude/Error de procesamiento** | Alta | Financiero |
| **2** | Cobro de mantención post-cierre | **Incumplimiento de instrucción** | Media | Administrativo |
| **3** | SMS malicioso y transferencias no autorizadas | **FRAUDE/SEGURIDAD** | **🔴 URGENTE** | **Cibernético** |
| **4** | App no funciona (error 503) | **Problema técnico** | Baja | Operativo |
| **5** | Rechazo de crédito sin explicación | **Falta de información** | Baja | Informativo |

---

## Notas importantes:

- **Reclamo #3 requiere acción inmediata**: contactar al banco por teléfono (no por app), bloquear la tarjeta y denunciar ante la policía.
- **Reclamo #1**: solicitar comprobante de transacción e iniciar disputa de cargo.
- **Reclamo #2**: presentar evidencia de solicitud de cierre.
- **Reclamo #4**: no es técnicamente un reclamo, es 

¿Te sirve esa respuesta para cargarla en un sistema? Probablemente no: el formato cambia cada vez. Ahora con **CRF + few-shot**:

In [19]:
# Misión 3b: CRF + few-shot + instrucciones negativas, con salida en JSON
import pandas as pd

ROL = "Actúas como analista senior de experiencia de clientes de un banco chileno."

CONTEXTO_Y_FORMATO = '''CONTEXTO: recibirás reclamos de clientes. Debes clasificarlos para derivarlos al área correcta.

FORMATO: responde SOLO con una lista JSON. Cada elemento debe tener estas claves:
- "id": número del reclamo
- "categoria": una de [Cobros, Fraude, Canales digitales, Productos, Crédito]
- "urgencia": una de [alta, media, baja]
- "area": una de [Operaciones, Seguridad, TI, Comercial, Riesgo]
- "resumen": máximo 12 palabras

REGLAS:
- NO inventes datos que no estén en el reclamo.
- NO agregues texto fuera del JSON.

EJEMPLOS:
Reclamo: "Me llegó un cargo de una tienda en Miami que nunca visité."
Respuesta: {"id": 0, "categoria": "Fraude", "urgencia": "alta", "area": "Seguridad", "resumen": "Cargo internacional desconocido en tarjeta"}

Reclamo: "El cajero se tragó mi tarjeta y no me dio el dinero."
Respuesta: {"id": 0, "categoria": "Cobros", "urgencia": "media", "area": "Operaciones", "resumen": "Cajero retuvo tarjeta y no entregó dinero"}

Reclamo: "Quiero cambiar mi plan de cuenta por uno sin costo de mantención."
Respuesta: {"id": 0, "categoria": "Productos", "urgencia": "baja", "area": "Comercial", "resumen": "Solicita cambio a plan sin costo de mantención"}
'''

def leer_json(texto):
    # Convierte la respuesta del modelo en datos; tolera texto extra alrededor del JSON.
    try:
        datos = json.loads(texto)
    except json.JSONDecodeError:
        trozo = re.search(r"\[.*\]|\{.*\}", texto, re.S)
        datos = json.loads(trozo.group(0)) if trozo else []
    if isinstance(datos, dict):
        listas = [v for v in datos.values() if isinstance(v, list)]
        datos = listas[0] if listas else [datos]
    return datos

def clasificar(lista_reclamos):
    numerados = "\n".join(f"{i}. {r}" for i, r in enumerate(lista_reclamos, 1))
    prompt = CONTEXTO_Y_FORMATO + "\nRECLAMOS A CLASIFICAR:\n" + numerados
    respuesta = preguntar(prompt, temperatura=0.1, instrucciones=ROL, json_salida=True)
    tabla = pd.DataFrame(leer_json(respuesta))
    columnas = [c for c in ["id", "categoria", "urgencia", "area", "resumen"] if c in tabla.columns]
    return tabla[columnas] if columnas else tabla

resultado = clasificar(reclamos)
resultado

,id,categoria,urgencia,area,resumen
0,1,Cobros,alta,Operaciones,"Doble cobro en supermercado, solicita reembolso"
1,2,Cobros,media,Operaciones,Sigue cobrando mantención tras cierre de cuenta
2,3,Fraude,alta,Seguridad,"SMS sospechoso, transferencias no autorizadas"
3,4,Canales digitales,media,TI,"App no accesible, error 503 post actualización"
4,5,Crédito,baja,Riesgo,Consulta rechazo de crédito de consumo


👉 **Prueba:** cambia la regla de urgencia (por ejemplo, *"todo lo que involucre dinero perdido es urgencia alta"*) y vuelve a ejecutar. Acabas de cambiar una **política de negocio** sin tocar el modelo.

## Misión 4 · Bonus: tu mini app 🚀
Con unas pocas líneas, el clasificador se convierte en un **formulario** que cualquier persona del equipo puede usar. Escribe un reclamo inventado y presiona el botón.

In [20]:
# Misión 4: una mini app con un cuadro de texto y un botón
import ipywidgets as widgets
from IPython.display import display, clear_output

caja = widgets.Textarea(placeholder="Escribe aquí el reclamo de un cliente (inventado)...",
                        layout=widgets.Layout(width="95%", height="90px"))
boton = widgets.Button(description="Clasificar reclamo", button_style="primary", icon="check")
salida = widgets.Output()

def al_hacer_click(_):
    with salida:
        clear_output()
        if not caja.value.strip():
            print("✍️ Escribe un reclamo primero.")
            return
        print("⏳ Clasificando...")
        tabla = clasificar([caja.value.strip()])
        clear_output()
        display(tabla)

boton.on_click(al_hacer_click)
display(widgets.HTML("<h3>🏦 Clasificador de reclamos</h3>"), caja, boton, salida)

HTML(value='<h3>🏦 Clasificador de reclamos</h3>')

Textarea(value='', layout=Layout(height='90px', width='95%'), placeholder='Escribe aquí el reclamo de un clien…

Button(button_style='primary', description='Clasificar reclamo', icon='check', style=ButtonStyle())

Output()

> **¿Quieres llevarlo más lejos?** El formulario de arriba ya es una mini aplicación. El mismo prompt (rol, formato y ejemplos) sirve en Claude.ai o en la consola de Anthropic, sin código.

## Para la puesta en común 💬
1. ¿Qué cambió entre la respuesta *zero-shot* y la de **CRF + few-shot**? ¿Cuál cargarías en un sistema?
2. ¿Qué te sorprendió del **costo** en la Misión 1? ¿Qué palanca lo baja más: un prompt más corto o un modelo más barato?
3. ¿Dejarías que este clasificador **derive reclamos sin revisión humana**? ¿En qué categoría pondrías siempre a una persona?
4. ¿Qué datos **nunca** pegarías en una herramienta de IA?
